<a href="https://colab.research.google.com/github/axelr004/AD_2026Q4_Grupo20/blob/main/datos_y_panda_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#importacion de librerias

In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import os
from IPython.display import display

pd.set_option("display.max_columns", None)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Librerías cargadas correctamente")

Pandas: 2.2.3
NumPy: 2.4.6
Librerías cargadas correctamente


#Comparar la temperatura máxima diaria y la precipitacióndiaria de Medellín y Madrid durante el año 2023.

#Se espera que Madrid presente una mayor variación estacional de temperatura, mientras que Medellín podría mostrar temperaturas más estables.

#También se esperan diferencias en la distribución mensual de las precipitaciones.

#se comprobarán mediante los siguoentes datos.

#periodo de estudio

In [51]:
FECHA_INICIO = "2023-01-01"
FECHA_FIN = "2023-12-31"
print("Periodo:", FECHA_INICIO, "hasta", FECHA_FIN)

Periodo: 2023-01-01 hasta 2023-12-31


#Crear carpeta de datos

In [52]:
os.makedirs("datos", exist_ok=True)
print("Carpeta de datos creada correctamente")

Carpeta de datos creada correctamente


#Descarga de los datos de Medellín y Madrid

In [53]:
URL_API = "https://archive-api.open-meteo.com/v1/archive"

def descargar_datos(ciudad, latitud, longitud):

    parametros = {
    "latitude": latitud,
    "longitude": longitud,
    "start_date": FECHA_INICIO,
    "end_date": FECHA_FIN,
    "daily": "temperature_2m_max,precipitation_sum",
    "temperature_unit": "celsius",
    "precipitation_unit": "mm",
    "timezone": "auto",
    "models": "era5" }

    respuesta = requests.get( URL_API,params=parametros, timeout=60 )
    respuesta.raise_for_status()
    datos = respuesta.json()

    tabla = pd.DataFrame({
        "fecha": datos["daily"]["time"],
        "temperatura_max_c": datos["daily"]["temperature_2m_max"],
        "precipitacion_mm": datos["daily"]["precipitation_sum"] })

    tabla.to_csv(
        f"datos/{ciudad}.csv", index=False )
    return tabla, datos


medellin, respuesta_medellin = descargar_datos(
    "medellin", 6.2442, -75.5812 )

madrid, respuesta_madrid = descargar_datos(
    "madrid", 40.4168, -3.7038 )

print("Datos descargados correctamente")

Datos descargados correctamente


#Mostrar las primeras filas

In [54]:
print("DATOS DE MEDELLÍN")
display(medellin.head())

print("DATOS DE MADRID")
display(madrid.head())

DATOS DE MEDELLÍN


,fecha,temperatura_max_c,precipitacion_mm
0,2023-01-01,23.8,14.5
1,2023-01-02,26.2,7.9
2,2023-01-03,23.5,9.6
3,2023-01-04,25.1,9.0
4,2023-01-05,25.7,6.9


DATOS DE MADRID


,fecha,temperatura_max_c,precipitacion_mm
0,2023-01-01,14.8,0.0
1,2023-01-02,12.3,1.0
2,2023-01-03,11.8,0.0
3,2023-01-04,12.3,0.0
4,2023-01-05,12.5,0.0


#Revisar dimensiones y estructura

In [55]:
print("MEDELLÍN")
print("Dimensiones:", medellin.shape)
medellin.info()
display(medellin.describe())

print("\nMADRID")
print("Dimensiones:", madrid.shape)
madrid.info()
display(madrid.describe())

MEDELLÍN
Dimensiones: (365, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   fecha              365 non-null    object 
 1   temperatura_max_c  365 non-null    float64
 2   precipitacion_mm   365 non-null    float64
dtypes: float64(2), object(1)
memory usage: 8.7+ KB


,temperatura_max_c,precipitacion_mm
count,365.000000,365.000000
mean,27.533151,7.621918
std,2.098776,7.863940
min,21.300000,0.000000
25%,26.200000,1.600000
50%,27.500000,5.500000
75%,29.300000,11.500000
max,32.800000,74.900000



MADRID
Dimensiones: (365, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   fecha              365 non-null    object 
 1   temperatura_max_c  365 non-null    float64
 2   precipitacion_mm   365 non-null    float64
dtypes: float64(2), object(1)
memory usage: 8.7+ KB


,temperatura_max_c,precipitacion_mm
count,365.000000,365.000000
mean,21.887671,1.197808
std,8.807900,4.571056
min,5.900000,0.000000
25%,14.200000,0.000000
50%,21.500000,0.000000
75%,28.500000,0.200000
max,39.300000,49.800000


#Revisar valores faltantes y duplicados

In [56]:
print("FALTANTES EN MEDELLÍN")
display(medellin.isna().sum())

print("FALTANTES EN MADRID")
display(madrid.isna().sum())

print("Duplicados Medellín:", medellin.duplicated().sum())
print("Duplicados Madrid:", madrid.duplicated().sum())
print( "Fechas duplicadas Medellín:", medellin["fecha"].duplicated().sum() )
print( "Fechas duplicadas Madrid:", madrid["fecha"].duplicated().sum() )

FALTANTES EN MEDELLÍN


,0
fecha,0
temperatura_max_c,0
precipitacion_mm,0


FALTANTES EN MADRID


,0
fecha,0
temperatura_max_c,0
precipitacion_mm,0


Duplicados Medellín: 0
Duplicados Madrid: 0
Fechas duplicadas Medellín: 0
Fechas duplicadas Madrid: 0


#convertir los tipos de datos

In [57]:
for tabla in [medellin, madrid]:

    tabla["fecha"] = pd.to_datetime( tabla["fecha"], errors="coerce" )
    tabla["temperatura_max_c"] = pd.to_numeric( tabla["temperatura_max_c"], errors="coerce" )
    tabla["precipitacion_mm"] = pd.to_numeric( tabla["precipitacion_mm"], errors="coerce" )

print("Tipos de datos de Medellín")
display(medellin.dtypes)

print("Tipos de datos de Madrid")
display(madrid.dtypes)

Tipos de datos de Medellín


,0
fecha,datetime64[ns]
temperatura_max_c,float64
precipitacion_mm,float64


Tipos de datos de Madrid


,0
fecha,datetime64[ns]
temperatura_max_c,float64
precipitacion_mm,float64


#Revisar el periodo descargado

In [58]:
print("MEDELLÍN")
print("Fecha inicial:", medellin["fecha"].min())
print("Fecha final:", medellin["fecha"].max())
print("Total de registros:", len(medellin))

print("\nMADRID")
print("Fecha inicial:", madrid["fecha"].min())
print("Fecha final:", madrid["fecha"].max())
print("Total de registros:", len(madrid))

MEDELLÍN
Fecha inicial: 2023-01-01 00:00:00
Fecha final: 2023-12-31 00:00:00
Total de registros: 365

MADRID
Fecha inicial: 2023-01-01 00:00:00
Fecha final: 2023-12-31 00:00:00
Total de registros: 365
